In [16]:
import pandas as pd

products = pd.read_csv("/kaggle/input/datasets/sadmanshafiq1611/product-details/product_details.csv")
# done = pd.read_csv("../data/output/products.csv")

# print(len(products))
# count = 0
# for index, row in done.iterrows():
#     if row["done"] == 1:
#         count += 1
# print(count)
print(products.head())
print(products.shape)
print(products.info())
print(products.describe())


                                    URL  \
0  https://www.amazon.com/dp/B0GJ5KNC27   
1  https://www.amazon.com/dp/B0FRN2Z4JF   
2  https://www.amazon.com/dp/B09C13PZX7   
3  https://www.amazon.com/dp/B0G12HGHGM   
4  https://www.amazon.com/dp/B0BX52C4LN   

                                               Title       Brand Rating  \
0  GravaStar Mercury M2 Wireless Gaming Mouse, 5 ...   GravaStar    4.5   
1  GravaStar Mercury M1 Pro Wireless Gaming Mouse...   GravaStar    4.5   
2  Razer Basilisk V3 Customizable RGB Wired Ergon...       Razer    4.6   
3  Logitech G PRO X2 SUPERSTRIKE Wireless Gaming ...  Logitech G    4.5   
4  HyperX Pulsefire Haste 2 – Wireless Gaming Mou...      HyperX    4.1   

    Price Review_Count                                          Image_URL  \
0   71.95        2,070  https://m.media-amazon.com/images/I/61M5yEjuIF...   
1   89.96        1,485  https://m.media-amazon.com/images/I/51BhB+GhmQ...   
2   34.98       10,516  https://m.media-amazon.com/images/I

# Preprocessing

In [17]:
print(len(products))
products = products.drop_duplicates(subset="URL", keep="first")
print(len(products))

1162
1152


In [18]:
print((products["Description"] == "not available").sum())
#products = products[products["Description"] != "not available"]

#print(len(products))



82


In [19]:
def convert_to_lower(text):
    text = str(text).lower()
    return text

combined_text = []

for index, row in products.iterrows():
    combined = row["Title"] + " " + row["Description"] + " " + row["Brand"]
    combined_text.append(convert_to_lower(combined))

products["Combined_Text"] = combined_text



    
    

# EDA

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english")
matrix = vectorizer.fit_transform(products["Combined_Text"])

similarity = cosine_similarity(matrix)

print(matrix.shape)



(1152, 10133)


In [25]:
def find_similar(URL, top_n):
    
    matching_rows = products.index[products["URL"] == URL]  
    product_row_number = matching_rows[0]
    scores_for_this_product = similarity[product_row_number]

    score_pairs = []
    for row_number in range(len(products)):
        if row_number != product_row_number: 
            score_pairs.append((row_number, scores_for_this_product[row_number]))
    
    score_pairs.sort(key=lambda x: x[1], reverse=True)

    source_title_words = set(convert_to_lower(products.loc[product_row_number, "Title"]).split())
    source_brand = products.loc[product_row_number, "Brand"]

    result = []
    for row_number, score in score_pairs[:top_n]:
        other_title_words = set(convert_to_lower(products.loc[row_number, "Title"]).split())
        shared_title_words = other_title_words.intersection(source_title_words)

        meaningful_shared_words = []
        for word in shared_title_words:
            if len(word) > 3:
                meaningful_shared_words.append(word)

        other_brand = products.loc[row_number, "Brand"]
        reason_parts = []
        if other_brand == source_brand:
            reason_parts.append("same brand")
        if len(meaningful_shared_words) > 0:
            top_shared_words = sorted(meaningful_shared_words)[:3]
            reason_parts.append("shared keywords: " + ", ".join(top_shared_words))
        if len(reason_parts) == 0:
            reason_parts.append("similar overall description")
        reason = "; ".join(reason_parts)

        result.append({
            "URL": products.loc[row_number, "URL"],
            "Ttle": products.loc[row_number, "Title"],
            "Similarity_Score": round(score, 3),
            "Reason": reason,
        })

    return pd.DataFrame(result)

example = products.loc[0, "URL"]
print("Products which are similar to:", products.loc[0, "Title"])
find_similar(example, 5)

Products which are similar to: GravaStar Mercury M2 Wireless Gaming Mouse, 5 Dynamic Lightsync RGB (White)


,URL,Ttle,Similarity_Score,Reason
0,https://www.amazon.com/dp/B0DRJ8FC3B,"GravaStar Mercury X Pro Wireless Gaming Mouse,...",0.275,"same brand; shared keywords: gaming, gravastar..."
1,https://www.amazon.com/dp/B0FRN2Z4JF,GravaStar Mercury M1 Pro Wireless Gaming Mouse...,0.238,"same brand; shared keywords: gaming, gravastar..."
2,https://www.amazon.com/dp/B0G39FDT9Q,"GravaStar Mercury M1 Pro Gaming Mouse, 78g Lig...",0.232,"same brand; shared keywords: gaming, gravastar..."
3,https://www.amazon.com/dp/B0DT43NNNF,"AULA WIN68 HE Mechanical Gaming Keyboard 60%, ...",0.146,shared keywords: gaming
4,https://www.amazon.com/dp/B0G4V76JFT,"Aula SC620 Wireless Gaming Mouse, Tri-Mode 2.4...",0.134,"shared keywords: gaming, mouse,, wireless"
